In [1]:
 
import numpy as np 
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import math
import pandas as pd 
import matplotlib.pyplot as plt

# import seaborn as sns
import sklearn.metrics as metrics
%matplotlib inline
import os
from pandas_ml import ConfusionMatrix

In [2]:
train = pd.read_csv('UNSW_NB15_training-set.csv')
test = pd.read_csv('UNSW_NB15_testing-set.csv')
combined_data = pd.concat([train, test]).drop(['id'],axis=1)

In [3]:
from sklearn.preprocessing import LabelEncoder,normalize
le1 = LabelEncoder()
le = LabelEncoder()

vector = combined_data['attack_cat']
print("attack cat:", set(list(vector))) # use print to make it print on single line 

combined_data['attack_cat'] = le1.fit_transform(vector)
combined_data['proto'] = le.fit_transform(combined_data['proto'])
combined_data['service'] = le.fit_transform(combined_data['service'])
combined_data['state'] = le.fit_transform(combined_data['state'])

vector = combined_data['attack_cat']

attack cat: {'Fuzzers', 'DoS', 'Analysis', 'Exploits', 'Normal', 'Backdoor', 'Shellcode', 'Reconnaissance', 'Generic', 'Worms'}


In [4]:
le1.inverse_transform([0,1,2,3,4,5,6,7,8,9])
combined_data.head(3)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,1,2,0,0,0,1,2,0,6,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,1,2,0,0,0,1,2,0,6,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,1,3,0,0,0,1,3,0,6,0


In [5]:
## OMITTED: For statistical feature removal

lowSTD = list(combined_data.std().to_frame().nsmallest(6, columns=0).index)
# this is stupid. suppose a feature has a 1.0 (spearman or pearson) correlation, OR conditional probability, when not 0.... That a very useful feature  

lowCORR = list(combined_data.corr().abs().sort_values('attack_cat')['attack_cat'].nsmallest(3).index) # .where(lambda x: x < 0.005).dropna()
# This might be stupid. A Deep MLP (feed forward neural net) may see patterns

drop = set( lowCORR + lowSTD)
drop = {'ackdat', 'ct_ftp_cmd', 'djit', 'is_ftp_login', 'is_sm_ips_ports', 'response_body_len', 'sjit', 'synack', 'tcprtt'}
# print(f'Before {combined_data.shape}')
combined_data_reduced=combined_data.drop(drop,axis=1)
# print(f'After {combined_data.shape}')

In [6]:
data_x = combined_data_reduced.drop(['attack_cat','label'], axis=1) # droped label
data_y = combined_data_reduced.loc[:,['label']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO

y_train = y_train.values.flatten()
y_test = y_test.values.flatten()

In [7]:
data_x2 = combined_data_reduced.drop(['attack_cat'], axis=1) # droped label
data_y2 = combined_data_reduced.loc[:,['label']]

X_train2, X_test2, y_train2, y_test2 = train_test_split(data_x2, data_y2, test_size=0.2, random_state=42)


In [8]:
Y_train = data_y2.values.flatten()
dict = {}
for i in Y_train:
    dict.update({i:dict.get(i,0)+1})
dict

{0: 93000, 1: 164673}

In [9]:
X_train.shape
data_y2.max()

label    1
dtype: int64

In [10]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape) # test is larger... good 
print(y_test.shape)
# y_train.min()

(206138, 33)
(206138,)
(51535, 33)
(51535,)


In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer

In [12]:
X = X_train
T = X_test
Y = y_train
C = y_test

scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)

traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)
testlabel = testlabel.flatten()

In [13]:
testlabel.shape

(51535,)

In [14]:

KNN = KNeighborsClassifier()
KNN.fit(traindata, trainlabel)

DT = DecisionTreeClassifier()
DT.fit(traindata, trainlabel)

RF = RandomForestClassifier(n_estimators=100)
RF.fit(traindata, trainlabel)

RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
                       max_depth=None, max_features='auto', max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=100,
                       n_jobs=None, oob_score=False, random_state=None,
                       verbose=0, warm_start=False)

In [15]:
np.seterr(invalid='ignore')

{'divide': 'warn', 'invalid': 'warn', 'over': 'warn', 'under': 'ignore'}

In [16]:

nums = 3  #分块数量

In [17]:
#-*- coding: utf8
from __future__ import division, print_function

import numpy as np

def _compute_centroids(X, assign, num_clusters):
    C = np.zeros(shape=(num_clusters, X.shape[1]), dtype='d')
    for k in range(num_clusters):

        if not (assign == k).any():
            continue

        K = X[assign == k]
        if K.ndim == 1:
            K = K[np.newaxis]
        C[k] = X[assign == k].mean(axis=0) 
    return C

def _surprisal_mat(X):
    #Some elements have zero prob, ingore and treat warnings
    with np.errstate(divide='ignore', invalid='ignore'):
        L = np.log2(X)
        L[np.isnan(L)] = 0
        L[np.isinf(L)] = 0
    return L

def _dist_all(X, C):
    S_x = _surprisal_mat(X)
    S_c = _surprisal_mat(C)
    
    D = (X * (S_x - S_c[:, np.newaxis,: ])).sum(axis=2).T
    return D

def _base_kmeans(X, C, n_iters=-1):
    
    num_clusters = C.shape[0]
    n = X.shape[0]

    C_final = C

    #KMeans algorithm
    cent_dists = None
    assign = None
    prev_assign = None
    best_shift = None

    iters = n_iters
    converged = False

    while iters != 0 and not converged:
        #assign elements to new clusters    
        D = _dist_all(X, C)
        assign = D.argmin(axis=1)
        
        #check if converged, if not compute new centroids
        if prev_assign is not None and not (prev_assign - assign).any():
            converged = True
        else: 
            C_final = _compute_centroids(X, assign, num_clusters)

        prev_assign = assign
        iters -= 1
    
    return C_final, assign

def cost(X, C, assign):
    cost = 0
    for k in set(assign):
        idx = assign == k
        cost += _dist_all(X[idx], C[k][np.newaxis]).sum()
    return cost

def klkmeans(X, num_clusters, n_iters=-1, n_runs=10):

    min_cost = float('+inf')
    best_C = None
    best_assign = None

    for _ in range(n_runs):
        assign = np.random.randint(0, num_clusters, X.shape[0])
        C = _compute_centroids(X, assign, num_clusters)

        C, assign = _base_kmeans(X, C, n_iters)
        clust_cost = cost(X, C, assign)

        if clust_cost < min_cost:
            best_C = C
            best_assign = assign

    return best_C, best_assign

if __name__ == '__main__':
    np.seterr(all='raise')
#     X = np.zeros((200, 1000))
#     X[0:100] = 1
#     X[100:200, 500:] = 1
#     X += 1e-20
    
#     X = (X.T / X.sum(axis=1)).T
#     C, assign = klkmeans(X, 2)
    
#     assert ((C.sum(axis=1) - 1) < 1e-10).all()
#     assert (assign[0:100] != assign[100:]).all()
    
    import os
#     dir_ = os.path.dirname('D:\mycode\CycleGAN_MetaLearning\src1_NSL-KDD')
#     fpath = os.path.dirname( './testdata.dat')
#     fpath = './testdata.dat'
#     X = np.genfromtxt(fpath)


    X = data_x2.values
    
    
    C, assign = klkmeans(X, nums) 
    assert len(set(assign)) == nums

  
#     for nums in range(10): 
#         nums += 1        
#         C, assign = klkmeans(X, nums)
#   #      C2, assign2 = klkmeans(X2, nums)
#         print(nums)
#         if len(set(assign)) == nums :
#             print("----"+str(nums))

In [18]:
assign_ = assign.reshape(len(assign),1)
X_ = np.concatenate((X,assign_),axis = 1)

# assign2_ = assign2.reshape(len(assign2),1)
# X2_ = np.concatenate((X2,assign2_),axis = 1)

In [19]:
assign

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)

In [20]:
print(X_.shape)
# print(X2_.shape)

(257673, 35)


In [21]:
CC={}
CT={}
for i in range(nums):
    
    X_df = X_[X_[...,-1]==i,:-1]
    y_df= X_[X_[...,-1]==i,-1]

    CC[i], CT[i], y_tr, y_te = train_test_split(X_df, y_df, test_size=0.2, random_state=42)
    print(CC[i].shape,end = " ")
    print(CT[i].shape)

(4480, 34) (1121, 34)
(105125, 34) (26282, 34)
(96532, 34) (24133, 34)


In [22]:
# np.set_printoptions(suppress=True)
# list = np.zeros(4)
# for i in assign:
#     list[i] += 1
# list

In [23]:
knn = []
dt = []
rf = []
for i in range(nums):
    knn_ = KNeighborsClassifier()
    knn_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    knn.append(knn_)
    
    dt_ = DecisionTreeClassifier()
    dt_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    dt.append(dt_)

    rf_ = RandomForestClassifier(n_estimators=100)
    rf_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    rf.append(rf_)
    
#     expected = CT[i][...,-1].flatten()
#     predicted = knn_.predict(CT[i][...,:-1])
#     # summarize the fit of the model
#     cm = ConfusionMatrix(expected, predicted)
#     print(cm)
#     try:
# #         np.errstate(divide="ignore")
#         cm.stats()
#     except:
#         continue
# #     print(cm.to_dataframe().loc['actual' = 'False', 'predicted' = 'False'])
#     print(cm(super.get(actual = 'False', predicted = 'False'))

In [24]:
# k= 3
# assign = np.random.randint(0, 4, 10)
# idx = assign == k
# print(idx)
# print(assign)


In [25]:
from pymoo.core.problem import Problem
class MyProblem(Problem):
    def __init__(self):
        self.i = i
        self.P1 = P1
        self.P2 = P2
        self.P3 = P3
        self.TP1 = TP1
        self.TP2 = TP2
        self.TP3 = TP3
        self.TN1 = TN1
        self.TN2 = TN2
        self.TN3 = TN3
        super().__init__(n_var=3,   # 变量数
                         n_obj=1,   # 目标数
                         n_constr=2,    # 约束数
                         xl=np.array([0,0,0]),     # 变量下界
                         xu=np.array([1,1,1]),   # 变量上界
                         )

    def _evaluate(self, x, out, *args, **kwargs):

        # 定义目标函数
#         f = dict([(key,[]) for key in range(4)])
#         f = dict.fromkeys(range(0,4),[])
#         print(i+1)
        
        f = 1 - (TP1 + TN1) * x[:,0] / P1  - (TP2 + TN2) * x[:,1] / P2 - (TP3 + TN3) * x[:,2] / P3 
    # 定义约束条件
        g1 = x[:,0] + x[:,1] + x[:,2] - 1
        g2 = - x[:,0] - x[:,1] - x[:,2] + 0.9
        # todo
        out["F"] = np.column_stack([f])
        out["G"] = np.column_stack([g1,g2])
        
#         print(cm1)
#         print(cm2)
#         print(cm3)
#         print()



In [26]:
# pd.crosstab(cm1._y_true, cm1._y_pred).reindex()[1][1]
# pd.crosstab(cm1._y_true, cm1._y_pred).iloc[1][1]

In [27]:

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.factory import get_sampling, get_crossover, get_mutation
from pymoo.optimize import minimize
# from example import MyProblem

# 定义遗传算法
algorithm = NSGA2(
    pop_size=40,
    n_offsprings=10,
    sampling=get_sampling("real_random"),
    crossover=get_crossover("real_sbx", prob=0.9, eta=15),
    mutation=get_mutation("real_pm", eta=20),
    eliminate_duplicates=True
)



Compiled modules for significant speedup can not be used!
https://pymoo.org/installation.html#installation

To disable this warning:
from pymoo.config import Config
Config.show_compile_hint = False



In [28]:
ans = []
for i in range(nums):
    print(i)
    
    predicted1 = knn[i].predict(CC[i][...,:-1])
    predicted2 = dt[i].predict(CC[i][...,:-1])
    predicted3 = rf[i].predict(CC[i][...,:-1])
    cm1 = ConfusionMatrix(CC[i][...,-1], predicted1)
    cm2 = ConfusionMatrix(CC[i][...,-1], predicted2)
    cm3 = ConfusionMatrix(CC[i][...,-1], predicted3)

    a = pd.crosstab(cm1._y_true, cm1._y_pred)
    TP1 = a.reindex()[0][0]
    TN1 = a.reindex()[1][1]
    P1 = cm1.population

    b = pd.crosstab(cm2._y_true, cm2._y_pred)
    TP2 = b.reindex()[0][0]
    TN2 = b.reindex()[1][1]
    P2 = cm2.population

    c = pd.crosstab(cm3._y_true, cm3._y_pred)
    TP3 = c.reindex()[0][0]
    TN3 = c.reindex()[1][1]
    P3= cm3.population
    print(cm1)
    print(cm2)
    print(cm3)
    print()
    
    res = minimize(MyProblem(),
                   algorithm,
                   ('n_gen', 40),
                   seed=1,
                   verbose=False
                   )
    ans.append(res)
    


0
Predicted  False  True  __all__
Actual                         
False        554    16      570
True          18  3892     3910
__all__      572  3908     4480
Predicted  False  True  __all__
Actual                         
False        570     0      570
True           0  3910     3910
__all__      570  3910     4480
Predicted  False  True  __all__
Actual                         
False        570     0      570
True           0  3910     3910
__all__      570  3910     4480

1
Predicted  False   True  __all__
Actual                          
False      20662   1490    22152
True        1174  81799    82973
__all__    21836  83289   105125
Predicted  False   True  __all__
Actual                          
False      22016    136    22152
True         369  82604    82973
__all__    22385  82740   105125
Predicted  False   True  __all__
Actual                          
False      21912    240    22152
True         266  82707    82973
__all__    22178  82947   105125

2
Predicted  False 

In [29]:
res.X

array([0.05791965, 0.09879949, 0.84088698])

In [30]:
for j in range(nums):
    
    print(ans[j].X)

[0.41417927 0.04995346 0.53583676]
[0.41417927 0.04995346 0.53583676]
[0.05791965 0.09879949 0.84088698]


In [31]:
ans[1].X[1]

0.04995345894608716

In [32]:
expected = testlabel
np.savetxt("Exp.txt", expected) 

predicted1 = KNN.predict(testdata)
predicted2 =  DT.predict(testdata)
predicted3 = RF.predict(testdata)

cm1 = ConfusionMatrix(expected, predicted1)
np.savetxt("Pre1.txt", predicted1) 

cm2 = ConfusionMatrix(expected, predicted2)
np.savetxt("Pre2.txt", predicted2) 

cm3 = ConfusionMatrix(expected, predicted3)
np.savetxt("Pre3.txt", predicted3) 

cm1.stats()

OrderedDict([('population', 51535),
             ('P', 32860),
             ('N', 18675),
             ('PositiveTest', 34510),
             ('NegativeTest', 17025),
             ('TP', 29325),
             ('TN', 13490),
             ('FP', 5185),
             ('FN', 3535),
             ('TPR', 0.8924223980523432),
             ('TNR', 0.7223560910307898),
             ('PPV', 0.8497536945812808),
             ('NPV', 0.7923641703377386),
             ('FPR', 0.27764390896921015),
             ('FDR', 0.15024630541871922),
             ('FNR', 0.10757760194765673),
             ('ACC', 0.8307946056078394),
             ('F1_score', 0.8705655336203058),
             ('MCC', 0.6282994913321212),
             ('informedness', 0.6147784890831329),
             ('markedness', 0.6421178649190193),
             ('prevalence', 0.6376249151062385),
             ('LRP', 3.214269678616685),
             ('LRN', 0.14892599824851663),
             ('DOR', 21.582999049319454),
             ('FOR', 

In [33]:
cm1.stats()

OrderedDict([('population', 51535),
             ('P', 32860),
             ('N', 18675),
             ('PositiveTest', 34510),
             ('NegativeTest', 17025),
             ('TP', 29325),
             ('TN', 13490),
             ('FP', 5185),
             ('FN', 3535),
             ('TPR', 0.8924223980523432),
             ('TNR', 0.7223560910307898),
             ('PPV', 0.8497536945812808),
             ('NPV', 0.7923641703377386),
             ('FPR', 0.27764390896921015),
             ('FDR', 0.15024630541871922),
             ('FNR', 0.10757760194765673),
             ('ACC', 0.8307946056078394),
             ('F1_score', 0.8705655336203058),
             ('MCC', 0.6282994913321212),
             ('informedness', 0.6147784890831329),
             ('markedness', 0.6421178649190193),
             ('prevalence', 0.6376249151062385),
             ('LRP', 3.214269678616685),
             ('LRN', 0.14892599824851663),
             ('DOR', 21.582999049319454),
             ('FOR', 

In [34]:
cm2.stats()

OrderedDict([('population', 51535),
             ('P', 32860),
             ('N', 18675),
             ('PositiveTest', 32641),
             ('NegativeTest', 18894),
             ('TP', 29530),
             ('TN', 15564),
             ('FP', 3111),
             ('FN', 3330),
             ('TPR', 0.8986609860012172),
             ('TNR', 0.8334136546184739),
             ('PPV', 0.9046904200238963),
             ('NPV', 0.8237535725627183),
             ('FPR', 0.1665863453815261),
             ('FDR', 0.09530957997610368),
             ('FNR', 0.10133901399878271),
             ('ACC', 0.8750169787523042),
             ('F1_score', 0.9016656234255965),
             ('MCC', 0.7302570602770089),
             ('informedness', 0.7320746406196912),
             ('markedness', 0.7284439925866146),
             ('prevalence', 0.6376249151062385),
             ('LRP', 5.394565706709332),
             ('LRN', 0.12159509678921017),
             ('DOR', 44.36499373047107),
             ('FOR', 0.

In [35]:
cm3.stats()

OrderedDict([('population', 51535),
             ('P', 32860),
             ('N', 18675),
             ('PositiveTest', 38866),
             ('NegativeTest', 12669),
             ('TP', 32564),
             ('TN', 12373),
             ('FP', 6302),
             ('FN', 296),
             ('TPR', 0.9909920876445526),
             ('TNR', 0.6625435073627844),
             ('PPV', 0.8378531364174343),
             ('NPV', 0.976635882863683),
             ('FPR', 0.33745649263721555),
             ('FDR', 0.16214686358256575),
             ('FNR', 0.009007912355447352),
             ('ACC', 0.8719705054817115),
             ('F1_score', 0.9080110420210245),
             ('MCC', 0.7295872571823246),
             ('informedness', 0.6535355950073369),
             ('markedness', 0.8144890192811172),
             ('prevalence', 0.6376249151062385),
             ('LRP', 2.9366514180834686),
             ('LRN', 0.013595955971710927),
             ('DOR', 215.99447837237426),
             ('FOR',

In [36]:
expectedEnd = np.empty(shape = [0,1]).flatten()
predictedEnd = np.empty(shape = [0,1]).flatten()
for i in range(nums):
    expected2 = CT[i][...,-1]
    testdata2 = CT[i][...,:-1]
    predicted2 = np.around(ans[i].X[0] * knn[i].predict(testdata2) +  ans[i].X[1] * dt[i].predict(testdata2) + ans[i].X[2] * rf[i].predict(testdata2))
#     predicted2 = np.around(0.09 * knn[i].predict(testdata2) +  0.3 * dt[i].predict(testdata2) + 0.2 * rf[i].predict(testdata2))
    expectedEnd = np.concatenate((expectedEnd, expected2), axis = 0)
    predictedEnd = np.concatenate((predictedEnd,predicted2), axis = 0)
    
    print(expectedEnd.shape)
    print(predictedEnd.shape)

cm = ConfusionMatrix(expectedEnd, predictedEnd)
expected = np.array(expectedEnd)
predicted = np.array(predictedEnd)

np.savetxt("AdassPre.txt", predicted) 
np.savetxt("AdassExp.txt", expected) 

cm.stats()


(1121,)
(1121,)
(27403,)
(27403,)
(51536,)
(51536,)


OrderedDict([('population', 51536),
             ('P', 32785),
             ('N', 18751),
             ('PositiveTest', 32664),
             ('NegativeTest', 18872),
             ('TP', 31483),
             ('TN', 17570),
             ('FP', 1181),
             ('FN', 1302),
             ('TPR', 0.960286716486198),
             ('TNR', 0.9370166924430697),
             ('PPV', 0.9638439872642665),
             ('NPV', 0.9310089020771514),
             ('FPR', 0.0629833075569303),
             ('FDR', 0.03615601273573353),
             ('FNR', 0.039713283513802045),
             ('ACC', 0.951820086929525),
             ('F1_score', 0.9620620635914987),
             ('MCC', 0.8960773114504456),
             ('informedness', 0.8973034089292677),
             ('markedness', 0.8948528893414178),
             ('prevalence', 0.6361572493014592),
             ('LRP', 15.246686046429042),
             ('LRN', 0.04238268521157098),
             ('DOR', 359.7385576283995),
             ('FOR', 0.